# [실습 과제] PyTorch로 유방암 데이터 분류 모델 구현하기

이번 과제에서는 튜토리얼에서 배운 PyTorch의 핵심 구성 요소들(`Dataset`, `DataLoader`, `nn.Module`)을 활용하여 위스콘신 유방암 데이터셋을 분류하는 딥러닝 모델을 직접 처음부터 끝까지 구현합니다.

**목표:**
1. `sklearn`의 데이터를 PyTorch `Dataset`으로 변환하기
2. `DataLoader`를 사용하여 데이터 배치 생성하기
3. `nn.Module`을 상속받아 직접 분류 모델 설계하기
4. 모델 학습 루프를 완성하고, 손실과 정확도 변화를 시각화하기

### 1. 라이브러리 임포트 및 데이터 로드

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# 데이터 로드
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

print(f"데이터 형태: {X.shape}")
print(f"레이블 형태: {y.shape}")
print(f"클래스 분포: {np.bincount(y)}")
print(f"\n데이터 설명:\n{cancer.DESCR[:450]}...")

### 2. 데이터 전처리 및 분할

모델이 데이터를 잘 학습하기 위해, 특성들의 스케일을 표준화하고 학습/테스트 세트로 분할합니다.

In [ ]:
# 학습 데이터와 테스트 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 데이터 스케일링 (중요!)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"학습 데이터 크기: {X_train.shape}")
print(f"테스트 데이터 크기: {X_test.shape}")

### 3. [TODO] PyTorch `Dataset` 생성

튜토리얼에서 배운 내용을 바탕으로, `__init__`, `__len__`, `__getitem__` 메서드를 완성하여 `BreastCancerDataset` 클래스를 구현하세요.

In [ ]:
class BreastCancerDataset(Dataset):
    def __init__(self, features, labels):
        # TODO: features와 labels를 올바른 PyTorch 텐서 타입으로 변환하여 self.features와 self.labels에 저장하세요.
        # 힌트: features는 FloatTensor, labels는 LongTensor
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        # TODO: 데이터셋의 전체 길이를 반환하도록 구현하세요.
        return len(self.features)

    def __getitem__(self, idx):
        # TODO: 주어진 인덱스(idx)에 해당하는 데이터와 레이블을 튜플 형태로 반환하세요.
        return self.features[idx], self.labels[idx]

# Dataset 인스턴스 생성
train_dataset = BreastCancerDataset(X_train, y_train)
test_dataset = BreastCancerDataset(X_test, y_test)

# 잘 생성되었는지 첫 번째 데이터 확인
print(f"첫 번째 학습 데이터: {train_dataset[0][0]}")
print(f"첫 번째 학습 레이블: {train_dataset[0][1]}")

### 4. [TODO] `DataLoader` 생성

학습 및 테스트 `Dataset`을 위한 `DataLoader`를 각각 생성하세요.
- 배치 크기는 64로 설정합니다.
- 학습용 `DataLoader`는 데이터를 섞도록 (`shuffle=True`) 설정하세요.

In [ ]:
BATCH_SIZE = 64

# TODO: train_dataset과 test_dataset을 위한 DataLoader를 생성하세요.
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# DataLoader 작동 확인
features_batch, labels_batch = next(iter(train_loader))
print(f"Feature batch shape: {features_batch.size()}")
print(f"Labels batch shape: {labels_batch.size()}")

### 5. [TODO] 신경망 모델 정의

`nn.Module`을 상속받아 분류 모델을 직접 설계하세요.
- 입력 특성 수: 30
- 출력 클래스 수: 2 (악성/양성)
- 최소 2개 이상의 은닉층(`nn.Linear`)과 `ReLU` 활성화 함수를 사용하세요.
- 예시 구조: `Input(30) -> Linear(16) -> ReLU -> Linear(8) -> ReLU -> Output(2)`

In [ ]:
class ClassificationModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(ClassificationModel, self).__init__()
        # TODO: 모델의 계층을 정의하세요.
        self.fc1 = nn.Linear(input_dim, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, output_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        # TODO: 입력 데이터 x가 계층들을 통과하는 순서를 정의하세요.
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# 모델 인스턴스화 및 구조 확인
INPUT_DIM = X_train.shape[1]
OUTPUT_DIM = 2
model = ClassificationModel(input_dim=INPUT_DIM, output_dim=OUTPUT_DIM)
print(model)

### 6. [TODO] 손실 함수와 옵티마이저 설정

- 손실 함수(`criterion`): 다중 클래스 분류에 적합한 `nn.CrossEntropyLoss`를 사용하세요.
- 옵티마이저(`optimizer`): `Adam` 옵티마이저를 사용하고, 모델의 파라미터와 학습률(learning rate) 0.001을 전달하세요.

In [ ]:
LEARNING_RATE = 0.001

# TODO: 손실 함수와 옵티마이저를 정의하세요.
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

### 7. 모델 학습 및 평가

이제 모든 준비가 끝났습니다. 아래 제공된 학습 및 평가 루프 코드를 실행하여 모델을 훈련시키세요. 코드가 어떻게 구성되어 있는지 각 주석을 꼼꼼히 읽어보세요.

In [ ]:
NUM_EPOCHS = 50
history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

for epoch in range(NUM_EPOCHS):
    # --- 학습 단계 ---
    model.train() # 모델을 학습 모드로 설정
    train_loss, train_correct, train_total = 0, 0, 0

    for features, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    avg_train_loss = train_loss / len(train_loader)
    train_accuracy = 100 * train_correct / train_total
    history['train_loss'].append(avg_train_loss)
    history['train_acc'].append(train_accuracy)

    # --- 평가 단계 ---
    model.eval() # 모델을 평가 모드로 설정
    test_loss, test_correct, test_total = 0, 0, 0

    with torch.no_grad(): # 평가 시에는 gradient 계산이 필요 없음
        for features, labels in test_loader:
            outputs = model(features)
            loss = criterion(outputs, labels)
            
            test_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()
            
    avg_test_loss = test_loss / len(test_loader)
    test_accuracy = 100 * test_correct / test_total
    history['test_loss'].append(avg_test_loss)
    history['test_acc'].append(test_accuracy)

    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}], Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.2f}%, Test Loss: {avg_test_loss:.4f}, Test Acc: {test_accuracy:.2f}%')

print("\nFinished Training!")

### 8. 학습 결과 시각화

에포크별 손실(loss)과 정확도(accuracy)의 변화를 시각화하여 학습이 잘 이루어졌는지 확인합니다.

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Loss vs. Epochs", "Accuracy vs. Epochs"))

# 손실 그래프
fig.add_trace(
    go.Scatter(x=list(range(1, NUM_EPOCHS + 1)), y=history['train_loss'], name='Train Loss', mode='lines'),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=list(range(1, NUM_EPOCHS + 1)), y=history['test_loss'], name='Test Loss', mode='lines'),
    row=1, col=1
)

# 정확도 그래프
fig.add_trace(
    go.Scatter(x=list(range(1, NUM_EPOCHS + 1)), y=history['train_acc'], name='Train Accuracy', mode='lines'),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(x=list(range(1, NUM_EPOCHS + 1)), y=history['test_acc'], name='Test Accuracy', mode='lines'),
    row=1, col=2
)

fig.update_layout(height=500, width=1000, title_text="Training and Validation History",
                  xaxis_title_text="Epochs", yaxis_title_text="Loss",
                  xaxis2_title_text="Epochs", yaxis2_title_text="Accuracy (%)")
fig.show()

### 과제 완료!

축하합니다! PyTorch를 사용하여 분류 모델을 성공적으로 구축, 학습, 평가했습니다. 

**추가적으로 생각해 볼 점:**
- 학습률(`LEARNING_RATE`)이나 배치 크기(`BATCH_SIZE`)를 변경하면 결과가 어떻게 달라질까요?
- 모델의 은닉층 수나 뉴런 수를 다르게 설계하면 성능이 더 좋아질까요?
- `ReLU`가 아닌 다른 활성화 함수(예: `Sigmoid`, `Tanh`)를 사용해 보세요.